In [1]:
import os, math
from pathlib import Path
import scanpy as sc
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy import stats
from scipy.stats import pearsonr, spearmanr, gaussian_kde
from scipy.sparse import issparse
import pickle

import gseapy as gp

/opt/conda/envs/scenv/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/opt/conda/envs/scenv/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/opt/conda/envs/scenv/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [2]:
# for clustermap
from scipy.cluster.hierarchy import linkage, leaves_list, dendrogram
from scipy.spatial.distance import pdist

import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch

In [11]:
sen_terms = pd.read_csv(
    "./data/sen_paper/2.4_sen_QuickGO_terms.tsv",
    header=None,
    names=["GO_id", "Ontology", "Term"],
    sep="\t",
)

In [12]:
ont_to_abb = {
    "biological_process": "BP",
    "molecular_function": "MF",
    "cellular_component": "CC",
}
sen_terms["Ontology"] = sen_terms["Ontology"].replace(ont_to_abb)

In [15]:
senescence_mapping = {
    # DDR & Telomeres
    "DNA damage response": "DNA Damage Response & Telomere Stress",
    "DNA damage response, signal transduction by p53 class mediator": "DNA Damage Response & Telomere Stress",
    "site of double-strand break": "DNA Damage Response & Telomere Stress",
    "regulation of telomere maintenance": "DNA Damage Response & Telomere Stress",
    "cellular response to gamma radiation": "DNA Damage Response & Telomere Stress",
    "negative regulation of DNA repair": "DNA Damage Response & Telomere Stress",
    "PML body": "DNA Damage Response & Telomere Stress",
    # Cell Cycle & Senescence Fate
    "negative regulation of cell cycle": "Cell Cycle Arrest & Senescence Fate",
    "regulation of cellular senescence": "Cell Cycle Arrest & Senescence Fate",
    # SASP & Inflammation
    "regulation of interleukin-6 production": "SASP Production & Inflammatory Signaling",
    "positive regulation of interleukin-6 production": "SASP Production & Inflammatory Signaling",
    "negative regulation of interleukin-6 production": "SASP Production & Inflammatory Signaling",
    "chemokine production": "SASP Production & Inflammatory Signaling",
    "positive regulation of chemokine production": "SASP Production & Inflammatory Signaling",
    "negative regulation of chemokine production": "SASP Production & Inflammatory Signaling",
    "inflammatory response": "SASP Production & Inflammatory Signaling",
    "positive regulation of inflammatory response": "SASP Production & Inflammatory Signaling",
    "positive regulation of canonical NF-kappaB signal transduction": "SASP Production & Inflammatory Signaling",
    "production of molecular mediator involved in inflammatory response": "SASP Production & Inflammatory Signaling",
    "cytokine activity": "SASP Production & Inflammatory Signaling",
    "metalloendopeptidase activity": "SASP Production & Inflammatory Signaling",
    # Antigen Presentation
    "antigen processing and presentation": "Antigen Presentation & Immune Recognition",
    "antigen processing and presentation of peptide antigen via MHC class I": "Antigen Presentation & Immune Recognition",
    "antigen processing and presentation of endogenous peptide antigen via MHC class I": "Antigen Presentation & Immune Recognition",
    "antigen processing and presentation of peptide antigen via MHC class II": "Antigen Presentation & Immune Recognition",
    "antigen processing and presentation of peptide or polysaccharide antigen via MHC class II": "Antigen Presentation & Immune Recognition",
    "antigen processing and presentation of endogenous antigen": "Antigen Presentation & Immune Recognition",
    "MHC class I protein complex": "Antigen Presentation & Immune Recognition",
    "MHC class Ib protein complex": "Antigen Presentation & Immune Recognition",
    "MHC class II protein complex": "Antigen Presentation & Immune Recognition",
    "MHC class II biosynthetic process": "Antigen Presentation & Immune Recognition",
    "regulation of MHC class II biosynthetic process": "Antigen Presentation & Immune Recognition",
    "positive regulation of MHC class II biosynthetic process": "Antigen Presentation & Immune Recognition",
    # Oxidative Stress
    "response to reactive oxygen species": "Oxidative Stress & ROS Homeostasis",
    "cellular response to reactive oxygen species": "Oxidative Stress & ROS Homeostasis",
    "cellular response to oxidative stress": "Oxidative Stress & ROS Homeostasis",
    "regulation of cellular response to oxidative stress": "Oxidative Stress & ROS Homeostasis",
    "hydrogen peroxide catabolic process": "Oxidative Stress & ROS Homeostasis",
    # Mitochondria
    "oxidative phosphorylation": "Mitochondrial Dynamics & Bioenergetics",
    "electron transport chain": "Mitochondrial Dynamics & Bioenergetics",
    "mitochondrial depolarization": "Mitochondrial Dynamics & Bioenergetics",
    "mitophagy": "Mitochondrial Dynamics & Bioenergetics",
    # Lysosomes
    "vacuolar acidification": "Lysosomal Function & Vacuolar Activity",
    "lysosomal lumen acidification": "Lysosomal Function & Vacuolar Activity",
    "vacuolar transmembrane transport": "Lysosomal Function & Vacuolar Activity",
    # Apoptosis & Immune Modulation
    "regulation of intrinsic apoptotic signaling pathway": "Apoptosis Regulation & Immune Modulation",
    "negative regulation of apoptotic signaling pathway": "Apoptosis Regulation & Immune Modulation",
    "positive regulation of apoptotic process": "Apoptosis Regulation & Immune Modulation",
    "negative regulation of apoptotic process": "Apoptosis Regulation & Immune Modulation",
    "response to type II interferon": "Apoptosis Regulation & Immune Modulation",
    "cellular response to cytokine stimulus": "Apoptosis Regulation & Immune Modulation",
    "immune response": "Apoptosis Regulation & Immune Modulation",
    "immune response-regulating cell surface receptor signaling pathway": "Apoptosis Regulation & Immune Modulation",
    "immune response-regulating signaling pathway": "Apoptosis Regulation & Immune Modulation",
    "negative regulation of inflammatory response": "Apoptosis Regulation & Immune Modulation",
}

# 2. Логический порядок категорий (для heatmaps, barplots, clustermap)
category_order = [
    "DNA Damage Response & Telomere Stress",
    "Cell Cycle Arrest & Senescence Fate",
    "SASP Production & Inflammatory Signaling",
    "Antigen Presentation & Immune Recognition",
    "Oxidative Stress & ROS Homeostasis",
    "Mitochondrial Dynamics & Bioenergetics",
    "Lysosomal Function & Vacuolar Activity",
    "Apoptosis Regulation & Immune Modulation",
    "Uncategorized",
]


# 3. Функция маппинга
def map_senescence_terms(df, term_col="Term", target_col="Term_cat"):
    """
    Маппит термины сенесцентности в функциональные категории.
    Обрабатывает точное совпадение строк, добавляет упорядоченный Categorical тип.
    """
    # Очистка от лишних пробелов
    df[term_col] = df[term_col].astype(str).str.strip()
    # Маппинг с fallback для непредвиденных терминов
    df[target_col] = df[term_col].map(senescence_mapping).fillna("Uncategorized")
    # Задание строгого порядка для визуализации
    df.loc[:, target_col] = pd.Categorical(
        df[target_col], categories=category_order, ordered=True
    )

    return df


# 4. Применение (пример)
sen_terms = map_senescence_terms(sen_terms)


In [4]:
with open("./data/sen_paper/2.4_sen_terms.pkl", "rb") as f:
    go_terms = pickle.load(f)


In [48]:
terms_to_save = [
    # антигенная презентация
    # "positive regulation of MHC class II biosynthetic process",
    "antigen processing and presentation of peptide antigen via MHC class I",
    "MHC class I protein complex",
    "antigen processing and presentation of endogenous antigen",  # ?
    "MHC class II protein complex",
    "antigen processing and presentation",
    "antigen processing and presentation of peptide antigen via MHC class II",
    #  апоптоз, имм. модуляция
    "negative regulation of apoptotic signaling pathway",
    "negative regulation of apoptotic process",
    "immune response-regulating signaling pathway",
    "response to type II interferon",  # ? мб в другую группу?
    "cellular response to cytokine stimulus",
    "negative regulation of inflammatory response",
    #  арест кл цикла и сенесцентная судьба
    "negative regulation of cell cycle",
    #  DDR, теломерный стресс
    "DNA damage response, signal transduction by p53 class mediator",
    "DNA damage response",
    "regulation of telomere maintenance",
    "negative regulation of DNA repair",
    "site of double-strand break",
    #  ф-ция лизосом и вакуолей
    "lysosomal lumen acidification",
    #  митохондрии, биоэнергетика
    "oxidative phosphorylation",
    "mitophagy",
    #  оксидативный стресс, гомеостаз ROS
    "hydrogen peroxide catabolic process",
    "response to reactive oxygen species",
    "cellular response to oxidative stress",
    #   SASP, воспалительный сигналинг
    "cytokine activity",
    "inflammatory response",
    "metalloendopeptidase activity",
    "production of molecular mediator involved in inflammatory response",
    "positive regulation of canonical NF-kappaB signal transduction",
    "chemokine production",
    "regulation of interleukin-6 production",
    #  прочее
    "negative regulation of autophagy",
    "stress-induced premature senescence",
    "oncogene-induced cell senescence",
    "G1 to G0 transition",
]
len(terms_to_save)

35

In [146]:
deg_df = pd.read_csv(
    "./data/sen_paper/2_pbDEGs_edgeR_NScFull.csv",
    index_col="gene_name",
)

In [147]:
# !!!! убираем из терминов гены, которые отсутствуют в данных

uniq_genes = set(deg_df.index)  # 21520 genes

filtered_go_terms = {}

for term, genes in go_terms.items():
    if term not in terms_to_save:
        continue

    genes_filtered = [g for g in genes if g in uniq_genes]

    if len(genes_filtered) > 0:
        filtered_go_terms[term] = genes_filtered

In [148]:
for term_cat in sen_terms.Term_cat.unique():
    if term_cat in (
        "SASP Production & Inflammatory Signaling",
        "Antigen Presentation & Immune Recognition",
    ):
        cat_terms = sen_terms[sen_terms.Term_cat == term_cat].Term.unique()
        cat_terms_dict = {
            k: filtered_go_terms[k] for k in cat_terms if k in filtered_go_terms
        }

In [149]:
results = []
# !!! ДЭГи нужно брать все, без фильтрации!!!
for t in tqdm(sorted(deg_df.Tissue.unique()), desc="Processing Tissues"):
    t_degs = deg_df[deg_df.Tissue == t]
    for ct in sorted(t_degs.cell_type.unique()):
        # print(f"✨----- {t} - {ct} -----")
        ct_degs = t_degs[t_degs.cell_type == ct].copy()
        min_fdr = ct_degs["FDR"][ct_degs["FDR"] > 0].min()
        fdr_clean = ct_degs["FDR"].replace(0, min_fdr)

        for term_cat in sen_terms.Term_cat.unique():
            if term_cat in (
                "SASP Production & Inflammatory Signaling",
                "Antigen Presentation & Immune Recognition",
            ):
                cat_terms = sen_terms[sen_terms.Term_cat == term_cat].Term.unique()
                cat_terms_dict = {
                    k: filtered_go_terms[k] for k in cat_terms if k in filtered_go_terms
                }

                pre_res = gp.prerank(
                    rnk=ct_degs.sort_values(
                        by=["logFC", "FDR"], ascending=[False, True]
                    )["logFC"],
                    outdir=None,
                    gene_sets=cat_terms_dict,
                    ascending=None,
                    min_size=2,
                    format=None,
                    verbose=False,
                )
                res_df = pre_res.res2d
                res_df["Name"] = "logFC"
                res_df["Term_cat"] = term_cat
                res_df["Tissue"] = t
                res_df["cell_type"] = ct
                results.append(res_df)


In [150]:
final_df = pd.concat(results, axis=0)
final_df.head(2)

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes,Term_cat,Tissue,cell_type
0,logFC,production of molecular mediator involved in i...,-0.45262,-1.320254,0.115304,0.705023,0.305296,8/20,16.22%,PDCD4;APPL1;PER1;RPS19;ZFP36;STAT3;VAMP2;HIF1A,SASP Production & Inflammatory Signaling,Colon,B cell
1,logFC,inflammatory response,-0.297727,-1.314687,0.045024,0.359817,0.311526,36/140,16.22%,APP;CD69;FOS;FOXP1;SMAD3;CCR6;IL10RB;CXCR4;NEK...,SASP Production & Inflammatory Signaling,Colon,B cell


In [151]:
final_df.shape

(711, 13)

In [152]:
signif_df = final_df[(final_df["FDR q-val"] < 0.05) & (final_df["NES"].abs() >= 1)]
signif_df.shape

(225, 13)

In [153]:
signif_df = signif_df.merge(sen_terms[["Term", "GO_id"]], on="Term", how="left")


In [154]:
term_mapping = {
    # --- Antigen Presentation & Immune Recognition ---
    "MHC class II protein complex": "MHC II complex",
    "antigen processing and presentation of peptide antigen via MHC class II": "MHC II Ag presentation",
    "antigen processing and presentation of endogenous antigen": "Endogenous Ag presentation",
    "antigen processing and presentation": "Ag processing & presentation",
    "MHC class I protein complex": "MHC I complex",
    "antigen processing and presentation of peptide antigen via MHC class I": "MHC I Ag presentation",
    # --- SASP Production & Inflammatory Signaling ---
    "metalloendopeptidase activity": "Metalloendopeptidase activity",
    "inflammatory response": "Inflammation",
    "production of molecular mediator involved in inflammatory response": "Inflammatory mediator production",
    "regulation of interleukin-6 production": "IL-6 production regulation",
    "positive regulation of canonical NF-kappaB signal transduction": "NF-κB pathway activation",
    "chemokine production": "Chemokine production",
    "cytokine activity": "Cytokine activity",
}
signif_df.Term = signif_df.Term.replace(term_mapping)

In [155]:
# 4. Обновляем столбец Term, добавляя ID в скобках
signif_df.loc[:, "Term"] = (
    signif_df.loc[:, "Term"] + " (" + signif_df.loc[:, "GO_id"] + ")"
)

In [156]:
# 2. Словарь категорий
cell_type_to_category = {
    # Immune & Hematopoietic
    "B cell": "Immune",
    "Plasma cell": "Immune",
    "CD4T": "Immune",
    "CD8T": "Immune",
    "Treg": "Immune",
    "NK": "Immune",
    "DC": "Immune",
    "Macrophage": "Immune",
    "Monocyte": "Immune",
    "Mast cell": "Immune",
    "Mono+mac": "Immune",
    # Epithelial
    "Epithelial cell": "Epithelial",
    "Basal cell": "Epithelial",
    "Club cell": "Epithelial",
    "Goblet cell": "Epithelial",
    "Respiratory tract goblet cell": "Epithelial",
    "Multiciliated epithelial cell": "Epithelial",
    "Pulmonary alveolar type 1 cell": "Epithelial",
    "Enterocyte": "Epithelial",
    "Paneth cell": "Epithelial",
    "Crypt stem cell": "Epithelial",
    "Transit amplifying cell": "Epithelial",
    # Vascular & Endothelial
    "Endothelial cell": "Endothelial",
    "Capillary endothelial cell": "Endothelial",
    "Vein endothelial cell": "Endothelial",
    "VE": "Endothelial",
    "Endothelial cell of lymphatic vessel": "Endothelial",
    "LE": "Endothelial",
    # Mesenchymal & Stromal
    "Fibroblast": "Mesenchymal & Stromal",
    "Pericyte": "Mesenchymal & Stromal",
    "Mural cell": "Mesenchymal & Stromal",
    "Adventitial cell": "Mesenchymal & Stromal",
    "Bronchial smooth muscle cell": "Mesenchymal & Stromal",
    "Adipocyte": "Mesenchymal & Stromal",
    # Specialized
    "Atrial Cardiomyocyte": "Tissue-Specific",
    "Neural cell": "Tissue-Specific",
    "Melanocyte": "Tissue-Specific",
}

# 4. Применение маппингов

signif_df.loc[:, "ct_category"] = signif_df["cell_type"].map(cell_type_to_category)

# 5. Задание строгого порядка категорий (критично для seaborn/matplotlib)
category_order = [
    "Immune",
    "Epithelial",
    "Endothelial",
    "Mesenchymal & Stromal",
    "Tissue-Specific",
]
signif_df.loc[:, "ct_category"] = pd.Categorical(
    signif_df["ct_category"], categories=category_order, ordered=True
)

signif_df.loc[signif_df["Tissue"].isin(["Heart_Cell", "Heart_Nuclei"]), "Tissue"] = (
    "Heart"
)

custom_tissue_order = [
    "Skin",
    "Lung",
    "Heart",
    "Colon",
    "Ileum",
    "Kidney",
]

In [157]:
unif_cts = {
    "Atrial Cardiomyocyte": "Cardiomyocytes",
    "Bronchial smooth muscle cell": "Bronchial SMCs",
    "CD4T": "CD4 T cells",
    "CD8T": "CD8 T cells",
    "Capillary endothelial cell": "Capillary endothelium",
    "DC": "Dendritic cells",
    "Endothelial cell of lymphatic vessel": "Lymphatic endothelium",
    "LE": "Lymphatic endothelium",
    "Macrophage": "Monocytes / macrophages",
    "Mono+mac": "Monocytes / macrophages",
    "Monocyte": "Monocytes / macrophages",
    "NK": "NK cells",
    "Respiratory tract goblet cell": "Goblet cells",
    "VE": "Vein endothelium",
    "Vein endothelial cell": "Vein endothelium",
    "gdT": "γδ T cells",
    "Pulmonary alveolar type 1 cell": "Type 1 pneumocytes",
    "B cell": "B cells",
    "Crypt stem cell": "Crypt stem cells",
    "Enterocyte": "Enterocytes",
    "Fibroblast": "Fibroblasts",
    "Mast cell": "Mast cells",
    "Transit amplifying cell": "Mid-crypt cells",
    "Endothelial cell": "Endothelial cells",
    "Mural cell": "Mural cells",
    "Adipocyte": "Adipocytes",
    "Neural cell": "Neural cells",
    "Goblet cell": "Goblet cells",
    "Paneth cell": "Paneth cells",
    "Plasma cell": "Plasma cells",
    "Epithelial cell": "Epithelial cells",
    "Adventitial cell": "Adventitial cells",
    "Basal cell": "Basal cells",
    "Club cell": "Club cells",
    "Multiciliated epithelial cell": "Multiciliated epithelial cells",
    "Melanocyte": "Melanocytes",
    "Pericyte": "Pericytes",
    "Treg": "Tregs",
}

# Более явный способ
signif_df["cell_type"] = signif_df["cell_type"].replace(unif_cts)

In [ ]:
# viz